# Tối ưu protocol chọn ngưỡng/trọng số trên validation 10k + stacking

- Validation mở rộng: 5.000 clone + 5.000 non-clone từ `validation_shard_0.parquet`.
- Model: M1 DiffMul gốc + M2 B5/B6 (best checkpoint).
- OOF 10-fold: isotonic calibration, ensemble weight grid, logistic stacking.
- Ngưỡng/trọng số chốt trên OOF; test chỉ đánh giá cuối kèm bootstrap CI + McNemar.


In [ ]:
import base64, io, os, shutil, subprocess, sys, tarfile, time
from pathlib import Path
WORK = Path('/kaggle/working')
CLEAN = WORK / 'clean_sel'
CLEAN.mkdir(parents=True, exist_ok=True)

def sh(cmd):
    print(f'\n$ {cmd}', flush=True)
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    with (WORK / 'live_progress.log').open('a', encoding='utf-8') as lf:
        for line in iter(p.stdout.readline, ''):
            print(line, end='', flush=True)
            lf.write(line); lf.flush()
    p.stdout.close()
    rc = p.wait()
    if rc != 0: raise RuntimeError(f'Fail rc={rc}: {cmd}')


In [ ]:
payload = '...'
repo_root = WORK / 'project'
repo_root.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(payload)), mode='r:gz') as tar:
    tar.extractall(repo_root)
os.chdir(repo_root)
print('Repo:', os.getcwd(), flush=True)


In [ ]:
import torch
print('CUDA:', torch.cuda.is_available(), flush=True)
if not torch.cuda.is_available(): raise SystemError('Chua bat GPU!')
sh('pip install -q -U transformers datasets scikit-learn scipy pyarrow')


In [ ]:
input_path = Path('/kaggle/input')
print('Inputs:', [p.name for p in input_path.iterdir()], flush=True)
data_raw = repo_root / 'data/raw/bigclonebench'
data_raw.mkdir(parents=True, exist_ok=True)

# validation shard + test parquet
for src_name, dest_name in (('validation_shard_0.parquet', 'validation_shard_0.parquet'), ('bcb_test.parquet', 'bcb_test.parquet')):
    cands = list(input_path.rglob(src_name))
    dest = data_raw / dest_name
    if cands and not dest.exists():
        sh(f'cp "{cands[0]}" "{dest}"')
    print(dest_name, dest.exists(), flush=True)
    assert dest.exists(), f'Thieu {dest_name}'

# checkpoint tu 2 dataset
ckpt_map = {
    'm1_best_checkpoint.pt': 'results/exp_m1_diffmul/best_checkpoint.pt',
    'm2_b5b6_best_checkpoint.pt': 'results/exp_m2_b5b6/best_checkpoint.pt',
}
found = {}
for f in input_path.rglob('*.pt'):
    if f.name in ckpt_map and f.name not in found: found[f.name] = f
for name, dest in ckpt_map.items():
    p = repo_root / dest
    p.parent.mkdir(parents=True, exist_ok=True)
    if name in found and not p.exists(): shutil.copy2(found[name], p)
    print(dest, p.exists(), flush=True)
    assert p.exists(), f'Thieu {dest}'

In [ ]:
print('=' * 70)
print('>>> BAT DAU OPTIMIZE SELECTION <<<')
print('=' * 70)
t0 = time.perf_counter()
sh('python scripts/optimize_selection.py --n-val 10000 --n-folds 10 --batch-size 32')
print(f'>>> XONG TRONG {(time.perf_counter()-t0)/60:.1f} PHUT', flush=True)
print((repo_root / 'results/exp_selection_opt/report.md').read_text(encoding='utf-8'), flush=True)

In [ ]:
WORK = Path('/kaggle/working')
out_dir = repo_root / 'results/exp_selection_opt'
for p in sorted(out_dir.rglob('*')):
    if p.is_file() and p.suffix in ('.json', '.md', '.npy', '.log'):
        name = 'sel_' + p.name
        shutil.copy2(p, CLEAN / name)
        shutil.copy2(p, WORK / name)
        print('  [SAVED]', name, flush=True)
log = WORK / 'live_progress.log'
if log.exists():
    shutil.copy2(log, CLEAN / 'sel_live_progress.log')
    shutil.copy2(log, WORK / 'sel_live_progress.log')
with tarfile.open(WORK / 'selection_opt_bundle.tar.gz', 'w:gz') as tar:
    for f in CLEAN.glob('*'): tar.add(f, arcname=f.name)
os.chdir(WORK)
for junk in ('project', 'data', 'clean_sel'):
    shutil.rmtree(WORK / junk, ignore_errors=True)
print('>>> XONG 100%!', flush=True)